# Cursul 4. Curățarea și integrarea datelor
**Fluxuri de lucru pentru știința datelor · Sociologie · 2026**

**Curățarea înseamnă să verificăm dacă datele sunt bune, înainte să le folosim.** În C3 le-am adus la aceeași formă; azi le verificăm, cu cinci întrebări:

| Verificarea | Întrebarea | În pandas |
|---|---|---|
| 1. Tipurile | Numerele sunt numere și codurile sunt text? | `dtypes` |
| 2. Duplicatele | Apare aceeași cheie de două ori? | `duplicated` |
| 3. Valorile imposibile sau neobișnuite | Există valori negative, procente peste 100, valori extreme? | `describe`, `nsmallest`, un filtru |
| 4. Lipsurile | Câte valori lipsesc și unde? | `count`, `isna` |
| 5. Codurile | Aceeași țară are același cod în toate sursele? | `replace` |

**Integrarea înseamnă să unim tabelele într-un tabel de analiză**, cu `pivot` și `merge`. Aici se pierd date fără nicio eroare, deci o singură regulă: **după fiecare `merge`, numărăm rândurile.**

| Pas | Ce facem | Timp |
|---|---|---|
| 1 | Deschidem tabelele din C3 | 5 min |
| 2 | Cele cinci verificări, pe indicatorii regionali | 25 min |
| 3 | Tabelul regiunilor: din lung în lat | 10 min |
| 4 | Perechile de regiuni | 15 min |
| 5 | Migrația și zborurile: coduri și acoperire | 15 min |
| 6 | Perechile de țări | 15 min |
| 7 | Salvăm și scriem deciziile | 5 min |

Tabelele finale merg tot în `data/processed`. În C5 le încărcăm într-o bază de date.

In [ ]:
import pandas as pd

## 1. Deschidem tabelele din C3
Toate sunt în `data/processed`, cu numele de coloane din C3.

In [ ]:
indicatori = pd.read_csv("data/processed/indicatori.csv")    # șapte indicatori, pe regiuni NUTS2
nume = pd.read_csv("data/processed/regiuni.csv")             # numele regiunilor
sci = pd.read_csv("data/processed/sci_regiuni.csv")          # SCI pe perechi de regiuni
indicatori.head()

## 2. Cele cinci verificări
### 2.1. Tipurile
**Anticipează:** ce tip are coloana `valoare`? Dar `regiune`?

In [ ]:
indicatori.dtypes

`valoare` este `float64`, adică număr cu zecimale; o valoare lipsă apare ca `NaN`. `regiune` și `indicator` sunt `object`, adică text. Dacă `valoare` ar fi `object`, ar fi un text ascuns printre numere (Eurostat scrie uneori `:` în loc de o valoare), iar calculele nu ar merge.

### 2.2. Duplicatele
Un rând e identificat de **regiune + indicator**: aceasta e cheia tabelului. Nicio cheie nu trebuie să apară de două ori.

In [ ]:
indicatori.duplicated(["regiune", "indicator"]).sum()    # câte rânduri repetă o cheie deja văzută

0: fiecare cheie apare o singură dată. (În SCI e altă situație, perechile duble din C3; le rezolvăm la pasul 4.)

### 2.3. Valorile imposibile sau neobișnuite
Un rezumat pe indicator: minimul, mediana, maximul.

In [ ]:
indicatori.groupby("indicator")["valoare"].describe().round(1)

**Interpretează:** populația minimă este 0. Care regiune e?

In [ ]:
populatie = indicatori[indicatori["indicator"] == "populatie"]
populatie.nsmallest(3, "valoare")          # cele mai mici trei valori

`NO0B` este Svalbard, arhipelagul norvegian: Eurostat raportează chiar 0. Nu e o greșeală. **Valorile neobișnuite nu se șterg automat**: întâi le căutăm în sursă.

**Exercițiu.** Studiile superioare, șomajul și rețelele sociale sunt procente, deci între 0 și 100. Câte valori sunt peste 100?

In [ ]:
# 1. Rândurile cu indicatorii în procente: studii_superioare, somaj, retele_sociale (cu isin)
# 2. Câte valori sunt peste 100? (o condiție, apoi .sum())

### 2.4. Lipsurile
Aria proiectului e dată de SCI. Eurostat are și regiuni care nu sunt în SCI, deci numărăm lipsurile doar pentru regiunile din SCI.

In [ ]:
regiuni_sci = sci["regiune_1"].unique()                          # lista regiunilor din SCI
print(len(regiuni_sci), "regiuni în SCI")

in_sci = indicatori[indicatori["regiune"].isin(regiuni_sci)]     # doar regiunile din SCI
in_sci.groupby("indicator")["valoare"].count()                  # câte au valoare, pe indicator

**Interpretează:** care indicator are cele mai multe lipsuri? Ce facem cu ele decidem după ce vedem tabelul regiunilor.

### 2.5. Codurile
Verificarea a cincea o facem la migrație și la zboruri (pasul 5), unde sursele scriu aceeași țară cu coduri diferite.

## 3. Tabelul regiunilor: din lung în lat
Trecem din formatul **lung** (un rând pe regiune și indicator) în formatul **lat** (un rând pe regiune, o coloană pe indicator). `pivot` face asta.

In [ ]:
valori = in_sci.pivot(index="regiune", columns="indicator", values="valoare")   # rânduri, coloane, celule
valori = valori.reset_index()                                                   # regiunea, din nou coloană
valori.head()

Pornim de la lista regiunilor din SCI și lipim numele și indicatorii. `how="left"` păstrează toate regiunile din SCI, chiar și pe cele fără date. Apoi regula integrării: numărăm rândurile.

In [ ]:
regiuni = pd.DataFrame({"regiune": regiuni_sci})                                     # cele 298 de regiuni din SCI
regiuni = regiuni.merge(nume[["regiune", "tara", "denumire"]], on="regiune", how="left")
regiuni = regiuni.merge(valori, on="regiune", how="left")

print(len(regiuni_sci), "->", len(regiuni), "regiuni")     # trebuie să fie același număr

Același număr: fiecare regiune a primit cel mult un rând. Dacă numărul ar crește, o regiune ar fi apărut de două ori în tabelul lipit; dacă ar scădea, am fi pierdut regiuni.

Câte valori lipsesc acum, pe coloană:

In [ ]:
regiuni.isna().sum()

Dacă am păstra doar regiunile complete, am pierde multe regiuni, iar pierderea nu e întâmplătoare: lipsurile sunt concentrate în câteva țări. **Decizia noastră: păstrăm toate regiunile** și alegem pentru fiecare analiză doar coloanele de care are nevoie.

## 4. Perechile de regiuni
În SCI, perechea A-B apare de două ori, o dată în fiecare ordine, cu aceeași valoare (am văzut în C3). Păstrăm o singură apariție: cea în care primul cod e mai mic alfabetic. Dispar și perechile unei regiuni cu ea însăși, pentru că un cod nu e mai mic decât el însuși.

**Anticipează:** câte perechi rămân din 298 de regiuni?

In [ ]:
perechi = sci[sci["regiune_1"] < sci["regiune_2"]]      # o singură ordine a fiecărei perechi
print(len(sci), "rânduri ->", len(perechi), "perechi")

298 × 297 / 2 = 44.253. Acum lipim indicatorii **de două ori**: o dată pentru capătul 1, o dată pentru capătul 2. `add_suffix("_1")` pune `_1` după fiecare nume de coloană, deci și `regiune` devine `regiune_1`, exact coloana după care lipim.

In [ ]:
atribute = regiuni.drop(columns=["tara", "denumire"])     # regiunea și cei șapte indicatori
atribute.add_suffix("_1").head(2)

In [ ]:
inainte = len(perechi)
perechi = perechi.merge(atribute.add_suffix("_1"), on="regiune_1", how="left")   # indicatorii capătului 1
perechi = perechi.merge(atribute.add_suffix("_2"), on="regiune_2", how="left")   # indicatorii capătului 2

print(inainte, "->", len(perechi), "perechi")      # regula integrării

## 5. Migrația și zborurile: coduri și acoperire
### 5.1. Codurile
Citim migrația din C3, cu aceeași grijă pentru Namibia (`NA`). Apoi verificarea a cincea, o capcană care nu dă nicio eroare: Eurostat scrie Grecia `EL` și Regatul Unit `UK`, iar SCI le scrie `GR` și `GB`. Dacă nu le aliniem, aceste țări se pierd la lipire, fără niciun mesaj.

In [ ]:
coduri = {"EL": "GR", "UK": "GB"}                    # codul Eurostat -> codul SCI

migratie = pd.read_csv("data/processed/migratie.csv", keep_default_na=False, na_values=[""])
migratie = migratie.replace({"tara_rezidenta": coduri, "cetatenie": coduri})    # în ambele coloane

zboruri = pd.read_csv("data/processed/zboruri.csv", keep_default_na=False, na_values=[""])
zboruri = zboruri.replace({"tara_1": coduri, "tara_2": coduri})

### 5.2. Acoperirea migrației
Nu toate țările trimit la Eurostat migranții pe fiecare cetățenie. Numărăm, pentru fiecare țară de rezidență, câte cetățenii au o valoare.

In [ ]:
acoperire = migratie.groupby("tara_rezidenta")["persoane"].count()    # câte cetățenii cu valoare, pe țară
acoperire.sort_values().head(12)                                    # cele care raportează cel mai puțin

**Interpretează:** unele țări au 0: nu raportează deloc cetățeniile. Perechile cu ele ar avea migrația incompletă. Lipsa se scrie în raport, nu se ascunde.

Păstrăm ca **raportoare** țările care dau cel puțin 20 de cetățenii.

In [ ]:
raportoare = acoperire[acoperire >= 20].index
print(len(raportoare), "țări raportoare:", list(raportoare))

## 6. Perechile de țări
Un rând va fi o pereche de țări, cu SCI, migrația în **ambele sensuri** și zborurile. SCI pe țări e în `country.csv`, din C1; și aici apare Namibia.

In [ ]:
tari = pd.read_csv("data/country.csv", keep_default_na=False)
tari = tari[["user_country", "friend_country", "scaled_sci"]]
tari.columns = ["tara_1", "tara_2", "sci"]

tari = tari[tari["tara_1"] < tari["tara_2"]]                                      # o singură ordine
tari = tari[tari["tara_1"].isin(raportoare) & tari["tara_2"].isin(raportoare)]    # ambele țări raportează
print(len(tari), "perechi de țări")

Lipim migrația de două ori, ca indicatorii la regiuni. Întâi cetățenii țării 1 care trăiesc în țara 2: redenumim coloanele migrației ca să se potrivească cu cheile din `tari`. Apoi invers.

In [ ]:
inainte = len(tari)

din_1_in_2 = migratie.rename(columns={"cetatenie": "tara_1", "tara_rezidenta": "tara_2", "persoane": "din_1_in_2"})
tari = tari.merge(din_1_in_2, on=["tara_1", "tara_2"], how="left")      # cetățenii țării 1 din țara 2

din_2_in_1 = migratie.rename(columns={"cetatenie": "tara_2", "tara_rezidenta": "tara_1", "persoane": "din_2_in_1"})
tari = tari.merge(din_2_in_1, on=["tara_1", "tara_2"], how="left")      # cetățenii țării 2 din țara 1

print(inainte, "->", len(tari), "perechi")      # regula integrării

SCI nu are sens: legătura RO-IT e aceeași cu IT-RO. Îl comparăm deci cu migrația **în ambele sensuri, adunată**. Dacă unul dintre sensuri lipsește, suma rămâne lipsă: nu numim „mică” o migrație pe care nu o cunoaștem.

La zboruri, fiecare țară își raportează pasagerii, deci perechea apare de două ori (am văzut în C3). Păstrăm valoarea raportată de țara 1.

In [ ]:
tari["migranti"] = tari["din_1_in_2"] + tari["din_2_in_1"]      # o valoare lipsă în sumă dă lipsă

inainte = len(tari)
tari = tari.merge(zboruri, on=["tara_1", "tara_2"], how="left")   # pasagerii raportați de țara 1
print(inainte, "->", len(tari), "perechi")

**Exercițiu.** Perechile României, de la cei mai mulți migranți la cei mai puțini. `|` înseamnă „sau”: România poate fi oricare dintre cele două capete.

In [ ]:
# 1. Perechile în care tara_1 este "RO" sau tara_2 este "RO" (două condiții legate cu |)
# 2. Ordonează după migranti, de la cel mai mare, și afișează primele 10

**Anticipează pentru C5:** perechile cu mulți migranți sunt și cele cu SCI mare? Dar cele cu mulți pasageri?

## 7. Salvăm și scriem deciziile
Trei tabele de analiză, în `data/processed`:

In [ ]:
regiuni.to_csv("data/processed/regiuni_nuts2.csv", index=False)       # un rând pe regiune
perechi.to_csv("data/processed/perechi_regiuni.csv", index=False)     # un rând pe pereche de regiuni
tari.to_csv("data/processed/perechi_tari.csv", index=False)           # un rând pe pereche de țări

### Jurnalul deciziilor
Orice curățare înseamnă decizii. Le scriem, cu motivul lor, ca oricine să le poată verifica (în proiect, în `README.md`).

| Tema | Decizia noastră | De ce |
|---|---|---|
| Valori neobișnuite (populația 0 la Svalbard) | păstrate | sunt raportate așa de sursă |
| Regiuni fără date | păstrate | lipsurile nu sunt întâmplătoare |
| Perechi duble A-B / B-A | o singură apariție | aceeași valoare, de două ori |
| Coduri EL/UK | aliniate la GR/GB | altfel pierdem țări la lipire |
| Țări care nu raportează migrația | scoase din perechile de țări | nu cunoaștem migrația lor |
| Zboruri raportate de ambele țări | valoarea raportată de țara 1 | o singură valoare pe pereche |

### Exercițiul echipei
1. Din ce țări sunt regiunile fără PIB?

In [ ]:
# 1. Regiunile în care pib_locuitor lipsește (isna)
# 2. Câte sunt, pe țară (value_counts pe coloana tara)

2. Găsiți perechea de țări a coridorului vostru în `tari`. Are migrația în ambele sensuri? Are zboruri? Dacă nu, de ce?

In [ ]:
# Perechea coridorului vostru: tara_1 și tara_2, în ordine alfabetică (de exemplu "IT" și "RO")

## Ce ducem mai departe
În `data/processed` avem trei tabele de analiză: `regiuni_nuts2.csv`, `perechi_regiuni.csv` și `perechi_tari.csv`. În C5 le încărcăm într-o bază de date și le agregăm.

**Poți explica fără cod?** Care sunt cele cinci verificări ale curățării? Ce identifică un rând în fiecare tabel? De ce păstrăm o singură ordine a perechii? Ce face `how="left"`? De ce numărăm rândurile după fiecare `merge`? De ce o sumă cu o valoare lipsă rămâne lipsă?